# 04_sentiment_analysis.ipynb

Train a clause‑level sentiment classifier and save the model.


In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import sys
from pathlib import Path

# Ensure project root is in sys.path so 'src' is always importable
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import joblib
from src.train import build_feature_pipeline, train_sentiment_classifier, generate_sentiment_label
from src.preprocessing import split_clauses

# Load preprocessed reviews
csv_path = ROOT_DIR / 'data' / 'processed' / 'reviews_preprocessed.csv'
df = pd.read_csv(csv_path)

# Sample for training sentiment if dataset is very large (e.g. 50,000 reviews)
sample_df = df.sample(n=min(50000, len(df)), random_state=42) if len(df) > 50000 else df

# Generate clause‑level data
clauses = []
labels = []
for text in sample_df['review_processed'].dropna():
    for clause in split_clauses(str(text)):
        if len(clause.split()) >= 2:  # keep informative clauses
            clauses.append(clause)
            labels.append(generate_sentiment_label(clause))

print(f'Extracted {len(clauses):,} clauses for sentiment training')

# Train sentiment classifier (default SVM)
vectorizer, model = train_sentiment_classifier(clauses, labels)

# Ensure models directory exists and save artifacts
models_dir = ROOT_DIR / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(vectorizer, models_dir / 'tfidf_vectorizer_sentiment.joblib')
joblib.dump(model, models_dir / 'sentiment_model.joblib')
print(f'Sentiment model saved to {models_dir}')

Extracted 79,607 clauses for sentiment training
Sentiment model saved to D:\Utkarsh VS Code\git\aspectify\models
